# Reply Mirror — Fraud Detection Multi-Agent System
### LangChain `create_agent` pattern + OpenRouter + Langfuse

## Cell 1 — Install
Pin `langchain==1.0.0` which is the version that ships `create_agent` in `langchain.agents`.

In [3]:
!pip install -q "langchain==1.0.0" langchain-core langchain-openai langchain-community langfuse ulid-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.4/476.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk

## Cell 2 — Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["OPENROUTER_API_KEY"]  = s.get_secret("OPENROUTER_API_KEY")
    os.environ["LANGFUSE_PUBLIC_KEY"] = s.get_secret("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = s.get_secret("LANGFUSE_SECRET_KEY")
    os.environ["LANGFUSE_HOST"]       = s.get_secret("LANGFUSE_HOST")
    os.environ["TEAM_NAME"]           = s.get_secret("TEAM_NAME")
    print("Secrets loaded from Kaggle.")
except Exception as e:
    print(f"Using defaults ({e})")

Secrets loaded from Kaggle.


In [ ]:
print("HOST  :", os.environ.get("LANGFUSE_HOST", "NOT SET"))
print("PUBLIC:", os.environ.get("LANGFUSE_PUBLIC_KEY", "NOT SET")[:20] + "...")
print("SECRET:", os.environ.get("LANGFUSE_SECRET_KEY", "NOT SET")[:20] + "...")
print("TEAM  :", os.environ.get("TEAM_NAME", "NOT SET"))
print("OPENR :", os.environ.get("OPENROUTER_API_KEY", "NOT SET")[:20] + "...")

## Cell 3 — Imports + Model + Langfuse
Exact tutorial import pattern:
```python
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
```

In [8]:
import os, json, re, pandas as pd, ulid
from datetime import timedelta
from pathlib import Path

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# ── New langfuse import — compatible with langchain==1.0.0 ───────────────────
from langfuse import Langfuse
from langfuse.langchain import CallbackHandler   # ← correct for new langfuse

import langchain
print("LangChain version:", langchain.__version__)

# ── Shared model ──────────────────────────────────────────────────────────────
model = ChatOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o-mini",
    temperature=0.2,
    max_tokens=800,
)

# ── Langfuse client ───────────────────────────────────────────────────────────
langfuse_client = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ["LANGFUSE_HOST"],
)

def generate_session_id():
    team = os.environ.get("TEAM_NAME", "team").replace(" ", "-")
    return f"{team}-{ulid.new().str}"

# def get_callback_handler():
#     return CallbackHandler(
#         public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
#         secret_key=os.environ["LANGFUSE_SECRET_KEY"],
#         host=os.environ["LANGFUSE_HOST"],
#     )

def get_callback_handler():
    return CallbackHandler()

print("Model and Langfuse ready.")

LangChain version: 1.0.0
Model and Langfuse ready.


## Cell 4 — Data Loader

In [9]:
def load_dataset(dataset_path: str) -> dict:
    """
    Loads all files from a dataset folder.
    Returns dict: transactions, users, sms, mails, locations, audio_files, level
    """
    base = Path(dataset_path)
    subs = [f for f in base.iterdir() if f.is_dir() and not f.name.startswith("__")]
    if subs:
        base = subs[0]

    data = {}

    tx = base / "transactions.csv"
    if tx.exists():
        df = pd.read_csv(tx)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        data["transactions"] = df
    else:
        data["transactions"] = pd.DataFrame()

    for key in ["users", "sms", "mails", "locations"]:
        p = base / f"{key}.json"
        data[key] = json.load(open(p, encoding="utf-8")) if p.exists() else []

    audio_dir = base / "audio"
    data["audio_files"] = sorted(audio_dir.glob("*.mp3")) if audio_dir.exists() else []

    n = len(data["users"])
    data["level"] = (
        "deus_ex"         if data["audio_files"] else
        "truman_show"     if n <= 3             else
        "brave_new_world" if n <= 7             else
        "deus_ex"
    )

    print(f"Level        : {data['level']}")
    print(f"Transactions : {len(data['transactions'])}")
    print(f"Users        : {n}")
    print(f"Audio files  : {len(data['audio_files'])}")
    return data

## Cell 5 — Load Dataset
> Update `DATASET_PATH` to your Kaggle input folder.

In [10]:
DATASET_PATH = "/kaggle/input/datasets/sadeghofoghi2/the-truman-show-train"

DATA = load_dataset(DATASET_PATH)

Level        : truman_show
Transactions : 80
Users        : 3
Audio files  : 0


## Cell 6 — Specialist Agents
Each specialist uses `create_agent(model=model, system_prompt="...")` — no tools.
Short prompts to minimize token usage while staying reliable.

In [ ]:
transaction_agent = create_agent(
    model=model,
    system_prompt="""You are a financial fraud analyst for MirrorPay.
Detect suspicious transactions: night activity (0-5am), amount vs salary,
new recipients, rapid sequences (3+ txns/hour), balance drain, e-commerce at night.
Never flag salary payments or regular monthly rent.
Return ONLY valid JSON:
{"flagged_transactions":[{"transaction_id":"...","risk_score":0.0,"flags":[]}],
"overall_risk":"low/medium/high","summary":"one sentence"}"""
)

user_profile_agent = create_agent(
    model=model,
    system_prompt="""You are a behavioral profiling expert for MirrorPay.
Assess phishing vulnerability and spending baseline from citizen profile data.
Return ONLY valid JSON:
{"phishing_risk":"high/medium/low","phishing_reasoning":"...",
"travel_frequency":"frequent/occasional/rare",
"monthly_salary_estimate":0,"behavioral_flags":[],"summary":"one sentence"}"""
)

communication_agent = create_agent(
    model=model,
    system_prompt="""You are a phishing detection analyst for MirrorPay.
Find phishing SMS/emails: fake domains (amaz0n, paypa1, ub3r, netfl1x),
urgent language, account verification requests, suspicious URLs.
Return ONLY valid JSON:
{"phishing_events":[{"channel":"sms/email","timestamp":"...","from":"...",
"severity":"high/medium","fake_domain":"..."}],
"total_attempts":0,"high_severity_count":0,
"most_targeted_service":"...","summary":"one sentence"}"""
)

correlation_agent = create_agent(
    model=model,
    system_prompt="""You are a fraud pattern analyst for MirrorPay.
Link phishing attacks to transactions within 72h windows.
Return ONLY valid JSON:
{"correlated_pairs":[{"phishing_timestamp":"...","transaction_id":"...",
"hours_gap":0,"amount":0,"combined_risk_score":0.0}],
"highest_risk_transactions":[],
"attack_narrative":"one sentence",
"overall_verdict":"clean/suspicious/high_risk"}"""
)

location_agent = create_agent(
    model=model,
    system_prompt="""You are a geospatial fraud analyst for MirrorPay.
Detect impossible travel: GPS pings vs in-person payment locations within 2h.
If no in-person payments: return {"skipped":true,"reason":"no in-person payments"}.
Return ONLY valid JSON:
{"anomalies":[{"transaction_id":"...","txn_location":"...",
"gps_city":"...","hours_gap":0,"severity":"high/medium/low"}],
"impossible_travel_detected":false,"summary":"one sentence"}"""
)

print("All 5 specialist agents ready.")

All 5 specialist agents ready.


## Cell 7 — @tool Wrappers
Each specialist is wrapped as a `@tool` so the FraudScorerAgent can call them.
Calling pattern mirrors the tutorial:
```python
response = agent.invoke({"messages": [HumanMessage(...)]})
return response["messages"][-1].content
```

In [12]:
@tool
def analyze_transactions(sender_id: str) -> str:
    """
    Analyzes all transactions for a sender and returns fraud signals.
    Use this first for every user investigation.

    Args:
        sender_id: The citizen sender ID to analyze

    Returns:
        JSON with flagged transactions, risk scores, flags.
    """
    df    = DATA["transactions"]
    users = DATA["users"]

    user_txns = df[df["sender_id"] == sender_id].sort_values("timestamp")
    if user_txns.empty:
        return json.dumps({"sender_id": sender_id, "flagged_transactions": []})

    # Salary reference
    iban_salary = {u["iban"]: u["salary"] for u in users if "iban" in u}
    sender_iban = user_txns["sender_iban"].dropna().iloc[0] \
                  if not user_txns["sender_iban"].dropna().empty else None
    monthly_salary = (iban_salary.get(sender_iban, 0) or 0) / 12

    # Rapid sequence detection
    times, ids = user_txns["timestamp"].tolist(), user_txns["transaction_id"].tolist()
    rapid_ids = set()
    for i in range(len(times)):
        w = [ids[j] for j in range(i, len(times))
             if times[j] - times[i] <= timedelta(hours=1)]
        if len(w) >= 3:
            rapid_ids.update(w)

    # Balance drain detection
    median_bal = user_txns["balance_after"].median()
    drain_ids = set(
        user_txns[user_txns["balance_after"] < median_bal * 0.05]["transaction_id"]
    ) if median_bal > 0 else set()

    seen, rows = set(), []
    for _, r in user_txns.iterrows():
        rid = r.get("recipient_id")
        rows.append({
            "transaction_id":           r["transaction_id"],
            "hour":                     r["timestamp"].hour,
            "amount":                   r["amount"],
            "type":                     r["transaction_type"],
            "is_new_recipient":         pd.notna(rid) and rid not in seen,
            "balance_after":            r["balance_after"],
            "amount_vs_monthly_salary": round(r["amount"] / monthly_salary, 2)
                                        if monthly_salary else None,
            "in_rapid_sequence":        r["transaction_id"] in rapid_ids,
            "balance_drain":            r["transaction_id"] in drain_ids,
            "description":              str(r.get("description", "")),
        })
        if pd.notna(rid):
            seen.add(rid)

    response = transaction_agent.invoke({"messages": [HumanMessage(
        f"Analyze transactions for sender {sender_id}. "
        f"Monthly salary ref: {monthly_salary:.0f}. "
        f"Data: {json.dumps(rows)}"
    )]})
    return response["messages"][-1].content


@tool
def profile_user(sender_id: str) -> str:
    """
    Builds behavioral risk profile for a user by sender_id.
    Always call before fraud scoring.

    Args:
        sender_id: The citizen sender ID to profile

    Returns:
        JSON with phishing risk, travel frequency, salary baseline.
    """
    users = DATA["users"]
    df    = DATA["transactions"]

    sender_txns = df[df["sender_id"] == sender_id]
    sender_iban = sender_txns["sender_iban"].dropna().iloc[0] \
                  if not sender_txns["sender_iban"].dropna().empty else None
    user = next((u for u in users if u.get("iban") == sender_iban), None)

    if not user:
        # Fallback: match by sender_id abbreviation
        parts = sender_id.split("-")
        if len(parts) >= 2:
            user = next((
                u for u in users
                if parts[1][:3].upper() in u.get("first_name", "").upper()[:4]
                or parts[0][:3].upper() in u.get("last_name",  "").upper()[:4]
            ), None)

    if not user:
        return json.dumps({"sender_id": sender_id, "found": False})

    response = user_profile_agent.invoke({"messages": [HumanMessage(
        f"Profile citizen for fraud risk. Data: {json.dumps(user)}"
    )]})
    return response["messages"][-1].content


@tool
def scan_communications(user_first_name: str) -> str:
    """
    Scans SMS and emails for phishing attacks targeting this user.

    Args:
        user_first_name: First name as it appears in messages (e.g. 'Alain')

    Returns:
        JSON with phishing events, severity, targeted services.
    """
    name = user_first_name.strip()
    user_sms  = [s["sms"][:250]  for s in DATA["sms"]
                 if name.lower() in s.get("sms",  "").lower()]
    user_mail = [m["mail"][:250] for m in DATA["mails"]
                 if name.lower() in m.get("mail", "").lower()]

    response = communication_agent.invoke({"messages": [HumanMessage(
        f"Find phishing attacks on user '{name}'. "
        f"SMS({len(user_sms)}): {json.dumps(user_sms[:10])} "
        f"Emails({len(user_mail)}): {json.dumps(user_mail[:5])}"
    )]})
    return response["messages"][-1].content


@tool
def correlate_fraud(sender_id: str, user_first_name: str) -> str:
    """
    Links phishing events to suspicious transactions within 72h windows.
    Call after analyze_transactions and scan_communications.

    Args:
        sender_id: The citizen sender ID
        user_first_name: First name for communication lookup

    Returns:
        JSON with correlated pairs, highest risk transactions, verdict.
    """
    txn_data  = analyze_transactions.invoke(sender_id)
    comm_data = scan_communications.invoke(user_first_name)

    response = correlation_agent.invoke({"messages": [HumanMessage(
        f"Correlate phishing to transactions for {user_first_name} ({sender_id}). "
        f"Transactions: {txn_data} | Communications: {comm_data}"
    )]})
    return response["messages"][-1].content


@tool
def check_locations(sender_id: str) -> str:
    """
    Detects impossible travel in in-person payments vs GPS pings.
    Automatically skipped if no in-person payments exist.

    Args:
        sender_id: The citizen sender ID

    Returns:
        JSON with location anomalies or skip confirmation.
    """
    df = DATA["transactions"]
    ip = df[
        (df["sender_id"] == sender_id) &
        (df["transaction_type"] == "in-person payment")
    ]

    if ip.empty:
        return json.dumps({"skipped": True, "reason": "no in-person payments"})

    locs = [
        l for l in DATA["locations"]
        if any(p in l.get("biotag", "") for p in sender_id.split("-")[:2])
    ]

    response = location_agent.invoke({"messages": [HumanMessage(
        f"Check location anomalies for {sender_id}. "
        f"Payments: {ip[['transaction_id','location','timestamp']].to_json(orient='records')} "
        f"GPS({len(locs)}): {json.dumps(locs[:15])}"
    )]})
    return response["messages"][-1].content


print("All @tool wrappers ready.")

All @tool wrappers ready.


## Cell 8 — FraudScorerAgent (Orchestrator)
Uses `create_agent(model=model, system_prompt="...", tools=[...])` — exact tutorial pattern.
This is the **only agent that calls the LLM as an orchestrator**.

In [ ]:
fraud_scorer_agent = create_agent(
    model=model,
    system_prompt="""You are a fraud investigator for MirrorPay.
Investigate each citizen using your tools in this order:
1. analyze_transactions(sender_id) — get transaction risk signals
2. profile_user(sender_id) — get phishing vulnerability and salary baseline
3. scan_communications(user_first_name) — detect phishing attacks
4. correlate_fraud(sender_id, user_first_name) — link phishing to transactions
5. check_locations(sender_id) — only if in-person payments may exist

Decision rules:
- combined_score >= 0.7 → fraudulent
- combined_score 0.5-0.7 + phishing context → fraudulent
- combined_score < 0.4 + no phishing → legitimate
- NEVER flag: salary payments, regular monthly rent
- New recipient alone is NOT enough — need 2+ signals

Return ONLY this JSON:
{"sender_id":"...","fraudulent_transactions":["id1","id2"],"reasoning":"brief"}""",
    tools=[
        analyze_transactions,
        profile_user,
        scan_communications,
        correlate_fraud,
        check_locations,
    ]
)

print("FraudScorerAgent ready.")

FraudScorerAgent ready.


## Cell 9 — Run per User + Orchestrator

In [14]:
def resolve_first_name(sender_id: str, dataset: dict) -> str:
    """Resolves user first name from sender_id via IBAN match or abbreviation fallback."""
    df, users = dataset["transactions"], dataset["users"]
    sender_txns = df[df["sender_id"] == sender_id]
    sender_iban = sender_txns["sender_iban"].dropna().iloc[0] \
                  if not sender_txns["sender_iban"].dropna().empty else None

    # Strategy 1: IBAN match
    iban_map = {u["iban"]: u.get("first_name", "") for u in users if "iban" in u}
    if sender_iban and sender_iban in iban_map:
        return iban_map[sender_iban]

    # Strategy 2: sender_id abbreviation match
    parts = sender_id.split("-")
    if len(parts) >= 2:
        for u in users:
            fn = u.get("first_name", "").upper()
            ln = u.get("last_name",  "").upper()
            if parts[1][:3] in fn[:4] or parts[0][:3] in ln[:4]:
                return u.get("first_name", "")

    return parts[0].capitalize()


def run_fraud_scorer(sender_id: str, user_first_name: str, session_id: str) -> dict:
    """Runs FraudScorerAgent for one citizen. Tracked by Langfuse."""
    langfuse_handler = get_callback_handler()

    response = fraud_scorer_agent.invoke(
        {"messages": [HumanMessage(
            f"Investigate sender_id='{sender_id}' "
            f"(first name: '{user_first_name}'). "
            f"Identify all fraudulent transaction IDs."
        )]},
        config={
            "callbacks": [langfuse_handler],
            "metadata": {"langfuse_session_id": session_id},
        }
    )

    final_text = response["messages"][-1].content

    # Parse JSON from response
    try:
        m = re.search(r"```json\s*(.*?)\s*```", final_text, re.DOTALL)
        parsed = json.loads(m.group(1) if m else final_text)
    except Exception:
        try:
            m = re.search(r"\{.*\}", final_text, re.DOTALL)
            parsed = json.loads(m.group(0)) if m else {}
        except Exception:
            parsed = {}

    return {
        "sender_id":               sender_id,
        "fraudulent_transactions": parsed.get("fraudulent_transactions", []),
        "reasoning":               parsed.get("reasoning", final_text[:200]),
    }


def run_orchestrator(dataset: dict) -> list:
    """Main entry point — investigates all citizens and writes output file."""
    session_id = generate_session_id()
    print(f"Session : {session_id}")
    print(f"Level   : {dataset['level']}")
    print("-" * 60)

    df = dataset["transactions"]
    SKIP = ("EMP", "ABIT", "DOM", "BTSWF")
    senders = [
        s for s in df["sender_id"].unique()
        if not any(str(s).startswith(p) for p in SKIP)
    ]

    print(f"Investigating {len(senders)} citizen(s)...\n")
    results = []

    for i, sid in enumerate(senders, 1):
        name = resolve_first_name(sid, dataset)
        print(f"[{i}/{len(senders)}] {sid} ({name})")

        result = run_fraud_scorer(sid, name, session_id)
        results.append(result)

        flagged = result.get("fraudulent_transactions", [])
        print(f"  Flagged : {len(flagged)} transaction(s)")
        for fid in flagged:
            print(f"    - {fid}")
        print(f"  Reason  : {result.get('reasoning', '')[:120]}")
        print()

    # Write output file
    fraud_ids, seen = [], set()
    for r in results:
        for fid in r.get("fraudulent_transactions", []):
            if fid not in seen:
                seen.add(fid)
                fraud_ids.append(fid)

    out_path = "/kaggle/working/fraudulent_transactions.txt"
    with open(out_path, "w") as f:
        f.write("\n".join(fraud_ids))

    print("=" * 60)
    print(f"Output   : {out_path}")
    print(f"Flagged  : {len(fraud_ids)} transactions")

    langfuse_client.flush()
    print(f"Traces   : sent to Langfuse | session: {session_id}")

    return results


print("run_orchestrator ready.")

run_orchestrator ready.


## Cell 10 — Run Pipeline

In [15]:
results = run_orchestrator(DATA)

Session : Hashtag-01KPBNQY4W1TV06FFNCF1NF89P
Level   : truman_show
------------------------------------------------------------
Investigating 3 citizen(s)...

[1/3] GRSC-KRLH-807-DIE-1 (Karl-Hermann)
  Flagged : 1 transaction(s)
    - 2d22a0cd-8297-4d67-bd84-9e1ceaba5b10
  Reason  : Transaction flagged due to new recipient and linked to high-severity phishing attempts.

[2/3] BRCH-TRCY-802-HAM-1 (Tracy)
  Flagged : 1 transaction(s)
    - 3ba7708b-367a-4637-9b52-d8fea85dc8e7
  Reason  : Transaction flagged due to night activity and new recipient, combined with medium phishing risk from recent high-severit

[3/3] RGNR-LNAA-7FF-AUD-0 (Alain)
  Flagged : 2 transaction(s)
    - 9caf7a6f-afa8-43a8-b0bb-8a81d16f8cc3
    - beaa4b03-0a2f-48a1-abb4-a5c7378ed046
  Reason  : Medium risk transactions flagged for night activity and new recipients, combined with multiple phishing attempts targeti

Output   : /kaggle/working/fraudulent_transactions.txt
Flagged  : 4 transactions
Traces   : sent to Lang

## Cell 11 — Validate Output
Checks the output file against the challenge validity rules.

In [16]:
with open("/kaggle/working/fraudulent_transactions.txt") as f:
    flagged_ids = [line.strip() for line in f if line.strip()]

all_ids     = DATA["transactions"]["transaction_id"].tolist()
pct_flagged = len(flagged_ids) / len(all_ids) * 100

print(f"Total transactions : {len(all_ids)}")
print(f"Flagged as fraud   : {len(flagged_ids)}")
print(f"Percentage flagged : {pct_flagged:.1f}%")
print()

# Challenge validity checks
assert len(flagged_ids) > 0,            "INVALID: no transactions reported"
assert len(flagged_ids) < len(all_ids), "INVALID: all transactions reported"

invalid_ids = [fid for fid in flagged_ids if fid not in all_ids]
if invalid_ids:
    print(f"WARNING: {len(invalid_ids)} invalid IDs found:")
    for bad in invalid_ids:
        print(f"  {bad}")
else:
    print("All flagged IDs are valid. Output is ready to submit.")

print("\nFlagged transaction IDs:")
for fid in flagged_ids:
    print(f"  {fid}")

Total transactions : 80
Flagged as fraud   : 4
Percentage flagged : 5.0%

All flagged IDs are valid. Output is ready to submit.

Flagged transaction IDs:
  2d22a0cd-8297-4d67-bd84-9e1ceaba5b10
  3ba7708b-367a-4637-9b52-d8fea85dc8e7
  9caf7a6f-afa8-43a8-b0bb-8a81d16f8cc3
  beaa4b03-0a2f-48a1-abb4-a5c7378ed046
